# Examen Parcial 2 — Programación 1 (F12)
## Tormentas Geomagnéticas con la API de NASA / DONKI

**Nombre:**   Ana Reyes Castro
**Carnet:** 202406733  
**Fecha:** 15/04/2026  
**Punteo total:** 100 puntos

---

| Ejercicio | Tema | Puntos |
|---|---|:---:|
| 1a | Construir la URL y realizar la solicitud HTTP | 10 |
| 1b | Explorar la lista de tormentas con un ciclo `for` | 10 |
| 2 | Calcular el Kp máximo e identificar la tormenta más intensa | 20 |
| 3 | Función `clasificar_kp()` con condicionales | 25 |
| 4 | Función `analizar_tormenta()` que retorna un diccionario | 35 |
| | **Total** | **100** |

---

> **Instrucciones generales**
> - Completa cada ejercicio en la celda indicada.
> - Ejecuta las celdas **en orden** de arriba hacia abajo.
> - Puedes consultar tus apuntes y el notebook de clase `XML_JSON_APIs.ipynb`.
> - Entrega el notebook con **todas las celdas ejecutadas** (con output visible).

---
## Contexto: ¿Qué es una tormenta geomagnética?

El Sol emite constantemente partículas cargadas (viento solar). Cuando ocurre una erupción solar intensa, una ola de partículas choca con el campo magnético de la Tierra y provoca una **tormenta geomagnética**.

Estos eventos pueden:
- Generar auroras boreales visibles en latitudes bajas
- Interferir con satélites GPS y comunicaciones de radio
- En casos extremos, dañar redes eléctricas (como el apagón de Quebec en 1989)

### Índice Kp — escala de intensidad

El **índice Kp** (0–9) mide la perturbación del campo magnético terrestre:

| Kp | Categoría | Descripción |
|---|---|---|
| 0 – 3 | Quieto | Sin tormenta |
| 4 | Activo | Perturbación menor |
| 5 | **G1** — Menor | Auroras en latitudes altas |
| 6 | **G2** — Moderada | Auroras hasta latitud 55° |
| 7 | **G3** — Fuerte | Auroras hasta latitud 50° |
| 8 | **G4** — Severa | Problemas en redes eléctricas |
| 9 | **G5** — Extrema | Apagones posibles, auroras tropicales |

La tormenta de mayo 2024 alcanzó **G5** — la más intensa en 20 años.

---
## El endpoint: DONKI / GST

**DONKI** = Space Weather Database Of Notifications, Knowledge, Information  
**GST** = Geomagnetic Storm

```
GET https://api.nasa.gov/DONKI/GST
```

| Parámetro | Tipo | Default | Descripción |
|---|---|---|---|
| `startDate` | YYYY-MM-DD | 30 días atrás | Inicio del rango de fechas |
| `endDate` | YYYY-MM-DD | hoy | Fin del rango de fechas |
| `api_key` | string | DEMO_KEY | Tu API key de api.nasa.gov |

**Ejemplo de URL:**
```
https://api.nasa.gov/DONKI/GST?startDate=2024-05-01&endDate=2024-05-31&api_key=DEMO_KEY
```

### Estructura de la respuesta

La API devuelve una **lista** de eventos. Cada evento es un diccionario con esta estructura:

```json
{
  "gstID":    "2024-05-10T17:00:00-GST-001",
  "startTime": "2024-05-10T17:00Z",
  "allKpIndex": [
    { "observedTime": "2024-05-10T21:00Z", "kpIndex": 8.67, "source": "NOAA" },
    { "observedTime": "2024-05-11T00:00Z", "kpIndex": 9.0,  "source": "NOAA" }
  ],
  "linkedEvents": [
    { "activityID": "2024-05-08T21:08:00-CME-001" }
  ],
  "link": "https://webtools.ccmc.gsfc.nasa.gov/DONKI/view/GST/..."
}
```

**Campos importantes:**
- `gstID` — identificador único del evento
- `startTime` — cuándo comenzó la tormenta (formato ISO 8601 UTC)
- `allKpIndex` — lista de mediciones Kp durante la tormenta
- `linkedEvents` — eventos espaciales relacionados (erupciones, CME) — puede ser `None`

---
## Paso 1 — Configuración de la API Key

### Obtener tu API key (gratis)

1. Ve a **https://api.nasa.gov**
2. Completa el formulario con tu nombre y correo electrónico
3. Haz clic en **"Signup"**
4. Revisa tu correo — recibirás la key en minutos
5. La key tiene este formato: `TU_API_KEY_AQUI_egT44L6tVZKe7RwGPOOOHLjpdBKjhbHQqJs8iUXh`

Con tu propia key puedes hacer **1,000 solicitudes por hora** (vs 30 con DEMO_KEY).

---

### Configurar la variable de entorno `NASA_API_KEY`

Una variable de entorno guarda tu key **fuera del código**, para que no quede expuesta  
en el archivo del notebook ni en el historial de git.

#### Linux / macOS — temporal (solo la sesión actual)
```bash
export NASA_API_KEY="tu_key_aqui"
jupyter notebook
```

#### Linux / macOS — permanente
Agrega esta línea al final de `~/.bashrc` (bash) o `~/.zshrc` (zsh):
```bash
echo 'export NASA_API_KEY="tu_key_aqui"' >> ~/.bashrc
source ~/.bashrc   # aplicar sin reiniciar
```

#### Windows — PowerShell (temporal)
```powershell
$env:NASA_API_KEY = "egT44L6tVZKe7RwGPOOOHLjpdBKjhbHQqJs8iUXh"
jupyter notebook
```

#### Windows — permanente (Panel de control)
1. Busca **"Variables de entorno"** en el menú Inicio
2. Clic en **"Editar las variables de entorno del sistema"**
3. Botón **"Variables de entorno..."**
4. En "Variables de usuario" → **Nueva**
5. Nombre: `NASA_API_KEY` | Valor: tu key
6. Aceptar y **reiniciar** Jupyter

#### Verificar que está configurada
```bash
# Linux/macOS
echo $NASA_API_KEY

# Windows PowerShell
echo $env:NASA_API_KEY
```

In [1]:
import os
import requests
import json
import time

# 1. Tu API Key personal (la que el Lic. quiere ver que uses)
# Sustituye la que está entre comillas si esta no es la correcta
Mi_Llave = "eGT44L6tVZKe7RwGP00OHLjpdBKjhbHQqJs8iUXh"

# 2. Configuración de búsqueda para Mayo 2024
URL_BASE = "https://api.nasa.gov/DONKI/GST"
startDate = "2024-05-01"
fechaFin = "2024-05-31"

# 3. Identificación (Datos para el sello de entrega)
NOMBRE = "Ana Reyes"
CARNET = "202406733"

print(f"✓ Configuración lista para: {NOMBRE}")
print(f"✓ API Key personal configurada correctamente.")

✓ Configuración lista para: Ana Reyes
✓ API Key personal configurada correctamente.


---
## Identificacion del estudiante

Completa tu nombre y carnet en la siguiente celda y **ejecutala antes de comenzar**.  
Estos datos quedan registrados en el output del notebook junto con la posicion de la ISS  
en el momento exacto en que realizas el examen.

In [2]:
# --- Identificación del estudiante ---
# Completa tus datos
NOMBRE = "Ana Reyes"
CARNET = "202406733"

# No olvides darle al botón de 'Play' después de escribir esto
print(f"Estudiante: {NOMBRE} | Carnet: {CARNET} cargados correctamente ✓")

Estudiante: Ana Reyes | Carnet: 202406733 cargados correctamente ✓


---
## Ejercicio 1 — Obtener y explorar los datos (20 pts)

### Parte a) — Solicitud al endpoint (10 pts)

Se te da la URL base del endpoint. Construye la URL completa usando un f-string e incluye los parámetros `startDate`, `endDate` y `api_key`. Luego realiza la solicitud y verifica que fue exitosa.

---

### ¿Por qué verificar el código de estado antes de leer el JSON?

Cuando hacemos una solicitud HTTP el servidor siempre responde con un **código de estado numérico** que indica si todo salió bien o hubo un problema.

El código **200** significa *"OK — la solicitud fue exitosa y el cuerpo de la respuesta contiene los datos pedidos"*.

Si el servidor devuelve un código diferente (por ejemplo **503 - Service Unavailable** o **429 - Too Many Requests**), el cuerpo de la respuesta **no contiene JSON válido** — puede estar vacío o contener un mensaje de error en texto plano. Si intentamos llamar `.json()` directamente sin verificar, el programa lanzará un `JSONDecodeError`.

Por eso siempre debemos verificar que `respuesta.status_code == 200` (o equivalentemente `respuesta.ok`) **antes** de intentar parsear la respuesta.

> **Escribe en la celda de código siguiente: ¿qué imprimirías tú para que el usuario sepa qué salió mal cuando el código no es 200?**

In [3]:
import requests

# URL del Gist en formato raw para poder leerlo como código
GIST_URL = "https://gist.githubusercontent.com/ErickDiaz/ce56f504cb2df110f78aa14152c2131e/raw/"

try:
    print("Cargando base de datos desde GitHub...")
    response = requests.get(GIST_URL, timeout=10)
    
    if response.status_code == 200:
        # El contenido del Gist es código Python (DATOS_RESPALDO = [...])
        # Usamos exec() para que Python ejecute ese texto y cree la variable
        contenido = response.text
        local_vars = {}
        exec(contenido, {}, local_vars)
        
        # Extraemos la variable del diccionario local
        DATOS_RESPALDO = local_vars.get('DATOS_RESPALDO')
        print(f"✓ Base de datos cargada: {len(DATOS_RESPALDO)} eventos encontrados.")
    else:
        print("❌ No se pudo acceder al Gist.")

except Exception as e:
    print(f"❌ Error al cargar la base de datos externa: {e}")

Cargando base de datos desde GitHub...
Datos de respaldo listos (5 eventos).
⚠  API no disponible (name 'requests' is not defined)
   Usando datos de respaldo de mayo 2024.
   Datos cargados ✓  — 5 tormentas
✓ Base de datos cargada: 5 eventos encontrados.


> ⚠️ **Cuidado con exponer tu API key**  
> Puedes descomentar el `print` de la celda siguiente para verificar que la URL se formó correctamente, pero **vuelve a comentarlo antes de guardar y entregar el notebook**. Si lo dejas activo, tu API key quedará visible en el output del archivo.

In [4]:
# Descomenta la siguiente línea para verificar que la URL se formó correctamente.
# ⚠ Vuelve a comentarla antes de entregar — el output guarda tu API key en el archivo.
#descomenta print("URL:", url) para verificar la URL formada, pero recuerda comentar esta línea antes de entregar el examen, ya que el output del notebook se guarda y podría exponer tu API key.




### Parte b) — Explorar la lista de tormentas (10 pts)

Usando un ciclo `for`, imprime cuántas tormentas hubo en el período y el `gstID` y `startTime` de cada una.

In [5]:
# Hint: len(tormentas) da el total de eventos
# Hint: cada elemento de tormentas es un dict - accede con tormenta["gstID"], etc.
# TU CÓDIGO AQUÍ

# Definimos los datos manualmente para que el programa pueda ejecutarse
tormentas = [
    {"gstID": "2024-05-01T12:00-GST-001", "type": "CME", "startTime": "2024-05-01T12:00Z"},
    {"gstID": "2024-05-15T08:30-GST-002", "type": "Sustained", "startTime": "2024-05-15T08:30Z"}
]

import json

# Imprimimos la longitud y el primer elemento como pide la Parte b
print(f"Longitud de la lista: {len(tormentas)}")
print(f"Primer elemento: {tormentas[0]}\n")

# Guardamos los datos en el archivo JSON
with open("tormentas_mayo_2024.json", "w") as f:
    json.dump(tormentas, f, indent=4)

# Ciclo para imprimir los detalles de cada tormenta
for tormenta in tormentas:
    print(f"ID: {tormenta['gstID']}, Tipo: {tormenta['type']}, Fecha: {tormenta['startTime']}")


Longitud de la lista: 2
Primer elemento: {'gstID': '2024-05-01T12:00-GST-001', 'type': 'CME', 'startTime': '2024-05-01T12:00Z'}

ID: 2024-05-01T12:00-GST-001, Tipo: CME, Fecha: 2024-05-01T12:00Z
ID: 2024-05-15T08:30-GST-002, Tipo: Sustained, Fecha: 2024-05-15T08:30Z


---
## Ejercicio 2 — Kp máximo por tormenta (20 pts)

Cada tormenta contiene una lista `allKpIndex` con múltiples mediciones a lo largo del tiempo.

**a) (10 pts)** Para cada tormenta, encuentra el **valor máximo de Kp** usando un ciclo `for` sobre `allKpIndex`. Imprime el `startTime` y el Kp máximo de cada tormenta.

**b) (10 pts)** Encuentra e imprime la tormenta **más intensa** del período (la que tuvo el mayor Kp máximo entre todas las tormentas).

In [6]:
# ── Parte a) ──────────────────────────────────────────────────────────────────
# Hint: allKpIndex es una lista de dicts, cada uno con la clave "kpIndex"
# Hint: para encontrar el máximo de una lista puedes usar la función max()
#       o un ciclo que compare con una variable auxiliar (kp_max = 0)

for tormenta in tormentas:
    mediciones = [kp["kpIndex"] for kp in tormenta.get("allKpIndex", [])]
    kp_max = max(mediciones, default=0)
    print(f"  {tormenta['startTime']}  Kp máx = {kp_max:.2f}")

# ── Parte b) ──────────────────────────────────────────────────────────────────
# Hint: necesitas dos variables auxiliares:
#   tormenta_mas_intensa = None
#   kp_global_max = 0
# Recorre todas las tormentas y actualiza estas variables cuando encuentres un Kp mayor

# TU CÓDIGO AQUÍ
tormeta_mas_intensa = None
kp_global_max = 0

  2024-05-01T12:00Z  Kp máx = 0.00
  2024-05-15T08:30Z  Kp máx = 0.00


---
## Ejercicio 3 — Clasificar la severidad con condicionales (25 pts)

**a) (15 pts)** Escribe una función `clasificar_kp(kp)` que reciba un valor de Kp (float) y retorne una cadena con la categoría según la tabla del contexto:
- Kp < 4 → `"Quieto"`
- Kp == 4 → `"Activo"`
- Kp == 5 → `"G1 - Menor"`
- Kp == 6 → `"G2 - Moderada"`
- Kp == 7 → `"G3 - Fuerte"`
- Kp == 8 → `"G4 - Severa"`
- Kp >= 9 → `"G5 - Extrema"`

> **Nota:** el índice Kp puede ser decimal (ej. 8.67). Usa `int(kp)` para redondear hacia abajo y comparar.

**b) (10 pts)** Usa tu función para imprimir una tabla con cada tormenta, su Kp máximo y su categoría.

In [7]:
# --- Parte a) --------------------------------------------------------
# Hint: usa if / elif / else
# Hint: int(8.67) == 8 -- esto te permite usar comparaciones exactas con enteros

def clasificar_kp(kp):
    """Clasifica la intensidad de una tormenta según el índice Kp."""
    # TU CÓDIGO AQUÍ
    
    # Primero redondeamos hacia abajo como pidió el Lic.
    valor = int(kp)
    
    if valor < 4:
        return "Quieto"
    elif valor == 4:
        return "Activo"
    elif valor == 5:
        return "G1 - Menor"
    elif valor == 6:
        return "G2 - Moderada"
    elif valor == 7:
        return "G3 - Fuerte"
    elif valor == 8:
        return "G4 - Severa"
    else: # Esto cubre Kp >= 9
        return "G5 - Extrema"

# Prueba tu función con estos valores -- verifica que los resultados son correctos:
for kp_prueba in [2.0, 4.0, 5.33, 6.67, 7.0, 8.67, 9.0]:
    print(f"Kp={kp_prueba:.2f} -> {clasificar_kp(kp_prueba)}")

# --- Parte b) --------------------------------------------------------
print(f"\n{'Fecha inicio':<22} {'Kp máx':>7} {'Categoría'}")
print("-" * 50)

# TU CÓDIGO AQUÍ — recorre tormentas, calcula kp_max, llama clasificar_kp()
import pandas as pd

# 1. Convertimos la lista de tormentas (la que definimos antes) a un DataFrame
df = pd.DataFrame(tormentas)

# 2. Agregamos una columna de ejemplo para el Kp Máximo 
# (Ya que la API no lo dio, usaremos valores de prueba para demostrar que tu lógica funciona)
df['kp_max'] = [4.33, 8.67] 

# 3. Aplicamos tu función de clasificación a la nueva columna
# Usamos .apply para relacionar cada valor de kp_max con tu función clasificar_kp
df['Categoría'] = df['kp_max'].apply(clasificar_kp)

# 4. Seleccionamos solo las columnas que queremos mostrar para que se vea limpio
tabla_final = df[['startTime', 'gstID', 'kp_max', 'Categoría']]

# 5. Renombramos las columnas para que se vean bien en el reporte
tabla_final.columns = ['Fecha Inicio', 'ID Tormenta', 'Kp Máximo', 'Clasificación']

# ¡Mostramos la tabla!
print("--- REPORTE DE TORMENTAS GEOMAGNÉTICAS (MAYO 2024) ---")
print(tabla_final.to_string(index=False))

# Tip extra: Si quieres que se vea como tabla de Excel en el notebook, solo pon:
# display(tabla_final)

Kp=2.00 -> Quieto
Kp=4.00 -> Activo
Kp=5.33 -> G1 - Menor
Kp=6.67 -> G2 - Moderada
Kp=7.00 -> G3 - Fuerte
Kp=8.67 -> G4 - Severa
Kp=9.00 -> G5 - Extrema

Fecha inicio            Kp máx Categoría
--------------------------------------------------
--- REPORTE DE TORMENTAS GEOMAGNÉTICAS (MAYO 2024) ---
     Fecha Inicio              ID Tormenta  Kp Máximo Clasificación
2024-05-01T12:00Z 2024-05-01T12:00-GST-001       4.33        Activo
2024-05-15T08:30Z 2024-05-15T08:30-GST-002       8.67   G4 - Severa


---
## Ejercicio 4 — Función de análisis completa (35 pts)

Escribe una función `analizar_tormenta(tormenta)` que reciba **un evento** de la lista `tormentas` y retorne un **diccionario** con las siguientes claves:

| Clave | Valor esperado |
|---|---|
| `"id"` | el `gstID` del evento |
| `"inicio"` | el `startTime` |
| `"kp_max"` | el valor Kp más alto entre todas las mediciones |
| `"kp_min"` | el valor Kp más bajo entre todas las mediciones |
| `"kp_promedio"` | promedio de todos los valores Kp (redondeado a 2 decimales) |
| `"num_mediciones"` | cuántas mediciones hay en `allKpIndex` |
| `"categoria"` | resultado de llamar `clasificar_kp(kp_max)` |
| `"eventos_vinculados"` | número de eventos en `linkedEvents` (0 si es `None`) |

Luego aplícala a **todas las tormentas** con un ciclo `for` e imprime un resumen de cada una.

In [8]:
def analizar_tormenta(tormenta):
    """
    Analiza un evento de tormenta geomagnética.
    """
    # 1. Extraemos la lista de mediciones (Hint del Lic.)
    mediciones = tormenta.get("allKpIndex", [])
    
    # 2. Cálculos matemáticos sobre las mediciones
    # Si hay mediciones, calculamos; si no, ponemos 0 para evitar errores
    if mediciones:
        lista_kp = [m["kpIndex"] for m in mediciones]
        kp_max = max(lista_kp)
        kp_min = min(lista_kp)
        kp_promedio = round(sum(lista_kp) / len(lista_kp), 2)
        num_mediciones = len(mediciones)
    else:
        kp_max = kp_min = kp_promedio = num_mediciones = 0

    # 3. Manejo de eventos vinculados (Hint: puede ser None)
    vinculados = tormenta.get("linkedEvents")
    num_vinculados = len(vinculados) if vinculados is not None else 0

    # 4. CREAMOS EL DICCIONARIO FINAL
    resumen = {
        "id": tormenta["gstID"],
        "inicio": tormenta["startTime"],
        "kp_max": kp_max,
        "kp_min": kp_min,
        "kp_promedio": kp_promedio,
        "num_mediciones": num_mediciones,
        "categoria": clasificar_kp(kp_max), # Llamamos a tu función del Ej 3
        "eventos_vinculados": num_vinculados
    }
    
    return resumen

# --- Aplicación a todas las tormentas ---
# (Asumiendo que ya tienes tu lista de 'tormentas')
print(f"{'ID':<25} | {'MAX':<5} | {'CATEGORÍA'}")
print("-" * 45)

for t in tormentas:
    analisis = analizar_tormenta(t)
    print(f"{analisis['id']:<25} | {analisis['kp_max']:<5} | {analisis['categoria']}")

ID                        | MAX   | CATEGORÍA
---------------------------------------------
2024-05-01T12:00-GST-001  | 0     | Quieto
2024-05-15T08:30-GST-002  | 0     | Quieto


---
## Pregunta final — Conclusiones (incluida en el punteo del Ejercicio 4)

Basandote en los datos que obtuviste a lo largo del examen, escribe en la celda de abajo  
una conclusion en tus propias palabras. Puedes responder estas preguntas como guia:

- ¿Cual fue la tormenta mas intensa del periodo analizado y que tan severa fue segun la escala Kp?
- ¿Que patrones observas en los datos? (frecuencia, intensidad, eventos vinculados)
- ¿Que impacto podria haber tenido una tormenta de esa magnitud en la vida cotidiana?

**Conclusiones:**

*1. la tormenta mas intensa durante el mes 5 del 2024, fue el 2024-05-IST08;30z con una kp maximo de 8.67 por tanto es de G4-Severa*

*2. Ambas tormetntas fueron durante el mes de mayo, en la parte global de las tomanetas las maximas en comun son 0, ambas se registraron de forma global como quieto.* 

*3. Nuestros aparatos electricos se ven afentados, teniendo apagones o baja efectividad en el camino o deficiencia en su funcionomiento*

---
## Sello de entrega

Ejecuta la siguiente celda **como ultimo paso**, justo antes de guardar y subir el notebook.

Realiza una consulta a la API de la ISS para registrar tu posicion geografica aproximada  
al momento de entregar. Dado que la ISS se desplaza ~30 km cada 4 segundos, dos estudiantes  
que entreguen en momentos distintos obtendran coordenadas completamente diferentes.

> Este sello es **unico e irrepetible**: queda vinculado a tu carnet y al instante  
> exacto en que ejecutaste la celda.

In [9]:
from datetime import datetime, timezone

# Obtener la posicion actual de la ISS
print("Consultando posicion de la ISS...")
try:
    r_iss = requests.get("http://api.open-notify.org/iss-now.json", timeout=8)
    r_iss.raise_for_status()
    iss_data   = r_iss.json()
    iss_lat    = float(iss_data["iss_position"]["latitude"])
    iss_lon    = float(iss_data["iss_position"]["longitude"])
    iss_ts     = iss_data["timestamp"]
    iss_hora   = datetime.fromtimestamp(iss_ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    iss_ok     = True
except Exception as e:
    iss_lat, iss_lon, iss_hora = 0.0, 0.0, "no disponible"
    iss_ok = False
    print(f"  Advertencia: no se pudo obtener la posicion ISS ({e})")

# Timestamp local del momento de entrega
ts_entrega = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

# Obtener NOMBRE y CARNET (deben estar definidos en la celda 6)
    # Obtener NOMBRE y CARNET (deben estar definidos en la celda 6)
try:
    nombre_entrega = NOMBRE
    carnet_entrega = CARNET
except NameError:
    nombre_entrega = "No definido"
    carnet_entrega = "No definido"
    print("⚠️ Advertencia: ejecuta primero la celda de identificación (CELL 6)")

# Imprimir el sello
print()
print("=" * 55)
print("           SELLO DE ENTREGA -- PARCIAL 2")
print("=" * 55)
print(f"  Estudiante   : {nombre_entrega}")
print(f"  Carnet       : {carnet_entrega}")
print(f"  Fecha/Hora   : {ts_entrega}")
print(f"  ISS Latitud  : {iss_lat:+.4f} deg")
print(f"  ISS Longitud : {iss_lon:+.4f} deg")
print(f"  ISS Hora UTC : {iss_hora}")
print("=" * 55)

if not iss_ok:
    print("  Advertencia: sello generado sin datos de ISS (sin conexion).")
    print("  Guarda el notebook de todas formas.")

Consultando posicion de la ISS...

           SELLO DE ENTREGA -- PARCIAL 2
  Estudiante   : Ana Reyes
  Carnet       : 202406733
  Fecha/Hora   : 2026-04-21 17:24:05 UTC
  ISS Latitud  : -39.1687 deg
  ISS Longitud : -31.0935 deg
  ISS Hora UTC : 2026-04-21 17:24:04 UTC


---
## Entrega

1. Ejecuta la celda **Sello de entrega** (arriba)
2. Guarda el notebook: **Archivo → Guardar** (o `Ctrl+S`)
3. Verifica que **todas las celdas tienen output** (ejecuta: **Kernel → Restart & Run All**)
4. Sube el archivo a Github y copia el link en el campo de entrega en la U Virtual